In [0]:
import json
import requests
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType

# ✅ Get run_id from task values
run_id = dbutils.jobs.taskValues.get(taskKey="Pre_validation", key="run_id", debugValue=None)
if not run_id:
    raise ValueError("❌ run_id could not be retrieved. Make sure it's passed from the first task.")

# ✅ Continue with API call
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()

headers = {"Authorization": f"Bearer {token}"}
url = f"{host}/api/2.1/jobs/runs/get?run_id={run_id}"
response = requests.get(url, headers=headers)
run_data = response.json()

# ✅ Extract and convert timestamps (correct field names)
start_ts = run_data.get("start_time")
end_ts = run_data.get("end_time")

start_time_str = datetime.fromtimestamp(start_ts / 1000.0).strftime("%Y-%m-%d %H:%M:%S") if start_ts else None
end_time_str = datetime.fromtimestamp(end_ts / 1000.0).strftime("%Y-%m-%d %H:%M:%S") if end_ts else None

# ✅ Define schema
schema = StructType([
    StructField("Start_Load_Date", StringType(), True),
    StructField("End_Load_Date", StringType(), True)
])

# ✅ Write to Delta
df = spark.createDataFrame([(start_time_str, end_time_str)], ["Start_Load_Date", "End_Load_Date"])
df.write.format("delta").mode("append").saveAsTable("oh_apm_stg.vendor_extracts.load_report_log_cpc_Stg_Ref")
